# Nonlinear Least Squares for SLAM Backends

The optimization machinery used by both [pose-graph SLAM](pose_graph_slam.ipynb) and [factor graphs](factor_graph.ipynb). Every SLAM backend boils down to this same recipe — only the residual $r(\xi)$ and the Jacobian $J$ change.

Three sections:

1. **The information matrix $\Omega$** — what it is, where it comes from for each sensor type, and how it weights the cost.
2. **Quadratic approximation of the cost** — Taylor expansion to derive the normal equations $H\,\delta x = b$.
3. **Gauss–Newton / Levenberg–Marquardt main loop** — the iterative algorithm that calls the linear system at each step.

The pose-graph notebook applies this framework to a specific residual $h(X_i, X_j) = X_i^{-1} X_j$; the factor-graph notebook applies it to a heterogeneous mix of factor types (between, prior, projection, range, bearing, …).

---


## 1. The Information Matrix $\Omega$

### 1.1 What it is

For each edge (measurement) you assume measurement noise with covariance:

$$
\Sigma_{ij} \in \mathbb{R}^{3\times 3}
$$

Then the **information matrix** is its inverse:

$$
\Omega_{ij} = \Sigma_{ij}^{-1}
$$

* Large covariance (uncertain measurement) ⟶ small information (weak weight)
* Small covariance (confident measurement) ⟶ large information (strong weight)

### 1.2 Why we need it

Not all constraints are equally reliable:

* Odometry often drifts (esp. wheel slip): higher uncertainty over time
* Loop closures can be very accurate, or sometimes wrong/outliers

$\Omega$ is how you tell the optimizer how much to "trust" each constraint.

---

### 1.3 How it appears in the pose-graph cost function

Pose graph SLAM is usually written as a weighted least squares problem:

$$
\min_{\{X_i\}} \sum_{(i,j)\in \mathcal{E}} e_{ij}^T \Omega_{ij} e_{ij}
$$

For our loop closure edge (3,0):

$$
\text{cost}_{30} = e_{30}^T \Omega_{30} e_{30}
$$

This is the **Mahalanobis distance**: it penalizes errors more strongly along directions you are confident about.

---

### 1.4 Choosing $\Sigma$ / $\Omega$ in practice (simple, common cases)

#### 1.4.1 Case A: independent noise in $(x,y,\theta)$

Often you assume:

$$
\Sigma = \mathrm{diag}(\sigma_x^2,\ \sigma_y^2,\ \sigma_\theta^2) \quad\Rightarrow\quad \Omega = \mathrm{diag}\left(\frac{1}{\sigma_x^2},\ \frac{1}{\sigma_y^2},\ \frac{1}{\sigma_\theta^2}\right)
$$

Example (numbers just to illustrate):

* loop closure believed accurate: $\sigma_x=\sigma_y=0.10$ m, $\sigma_\theta=0.05$ rad

$$
\Omega_{LC} = \mathrm{diag}(100,\ 100,\ 400)
$$

Then angular errors get penalized more than translation errors in this example (because $1/0.05^2 = 400$).

#### 1.4.2 Case B: correlated noise

If $(x,y,\theta)$ are correlated (common with scan matching), $\Sigma$ has off-diagonal terms, and $\Omega=\Sigma^{-1}$ is full. This rotates/scales the penalty ellipse in error space.

---

### 1.5 Where do the covariance values come from?

In practice you almost never know $\Sigma$ exactly. Three common strategies, in order of rigor:

1. **Inverse Hessian of the front-end optimizer** (Cramér–Rao / Laplace approximation). Whatever computed the relative pose — ICP, bundle adjustment, scan matcher — internally minimizes a cost function. At the solution, the inverse Hessian of that cost is a first-order approximation to $\Sigma$.
2. **Calibration from ground truth.** Drive/walk a known trajectory (motion capture, total station, or just a tape measure for short tracks). For each measurement, compute the residual against truth. Fit a zero-mean Gaussian; its sample covariance is your $\Sigma$.
3. **Hand-tuned heuristics.** Pick numbers that look reasonable and adjust until the optimizer behaves. Surprisingly common in production systems; the *relative* ratios between edge types matter much more than the absolute scale.

The first strategy — the **inverse-Hessian recipe** — produces almost every covariance in modern SLAM front ends, so it's worth unpacking once (§1.5.1) before walking through the per-sensor specifics.

#### 1.5.1 The inverse-Hessian recipe (and why it's the same formula everywhere)

Whenever a front end estimates a relative pose $\xi$ by minimizing some residual, the covariance of that estimate has the form

$$
\Sigma \;\approx\; \big(J^T\, W\, J\big)^{-1}.
$$

This is the **Laplace approximation** of the posterior, equivalently the inverse Fisher information of the MLE under Gaussian residual noise. The pieces:

* $r(\xi)$ — vector of residuals (point-to-point distances, reprojection errors, photometric differences, ...) as a function of the pose $\xi$.
* $J = \partial r / \partial \xi$, evaluated at the optimum $\hat\xi$.
* $W$ — inverse covariance of the per-residual noise (e.g., $W = \tfrac{1}{\sigma_{\text{pix}}^2} I$ for isotropic 1-pixel reprojection noise).
* $H = J^T W J$ — the Gauss–Newton approximation of the Hessian of $\tfrac12\, r(\xi)^T W\, r(\xi)$ at $\hat\xi$.

**Where the formula comes from.** Near the optimum, the cost is approximately quadratic,

$$
\tfrac12\, r(\xi)^T W\, r(\xi) \;\approx\; \tfrac12\, r(\hat\xi)^T W\, r(\hat\xi) + \tfrac12\,(\xi-\hat\xi)^T H (\xi-\hat\xi),
$$

so the likelihood of $\xi$ given the data is locally a Gaussian centered at $\hat\xi$ with covariance $H^{-1}$. That's exactly the $\Sigma$ we want.

**Minimal numerical example: 2D translation from three point matches.**

Imagine an ICP-style front end where the only unknown is a 2D translation $\xi = (x, y)$, and we have three correspondences:

Source: $s_1 = (1, 0),\ s_2 = (0, 1),\ s_3 = (-1, 0)$  
Target: $t_1 = (1.10, 0.05),\ t_2 = (0.05, 1.05),\ t_3 = (-0.85, 0.10)$

Per-correspondence residual:

$$
r_k(\xi) = s_k + \xi - t_k \;\in\; \mathbb{R}^2
$$

Stack all three to get $r \in \mathbb{R}^6$. The Jacobian is six rows of $I_2$:

$$
J = \frac{\partial r}{\partial \xi} = \begin{bmatrix} I_2 \\ I_2 \\ I_2 \end{bmatrix} \in \mathbb{R}^{6\times 2}
$$

Assume each point match has isotropic noise $\sigma_{\text{pt}} = 0.1\text{ m}$, so the per-residual covariance is $\sigma_{\text{pt}}^2 I_2$ and the stacked weight matrix is $W = \tfrac{1}{\sigma_{\text{pt}}^2} I_6 = 100\, I_6$. Then

$$
H \;=\; J^T W J \;=\; \frac{1}{\sigma_{\text{pt}}^2}\,(I_2 + I_2 + I_2) \;=\; 300\, I_2,
$$

$$
\Sigma \;=\; H^{-1} \;=\; \frac{1}{300}\, I_2 \;\approx\; \begin{bmatrix} 0.00333 & 0 \\ 0 & 0.00333 \end{bmatrix}.
$$

So $\sigma_x = \sigma_y = \sqrt{\sigma_{\text{pt}}^2 / 3} \approx 0.058\text{ m}$. The general rule of thumb falls right out: with $N$ matches each at noise $\sigma$, the translation estimate sharpens as

$$
\sigma_{\text{pose}} \;\approx\; \sigma\,/\sqrt{N}.
$$

Add more matches, $\Sigma$ shrinks. Use noisier matches, $\Sigma$ grows. Cluster all your matches along one direction and $J^T W J$ becomes rank-deficient orthogonal to that direction — $\Sigma$ blows up there, exactly capturing the "long featureless corridor" failure mode of ICP.

**Why §1.5.3, §1.5.4, §1.5.5 use the same equation.**

Each of those front ends is a nonlinear least squares problem with the same structure,

$$
\hat\xi \;=\; \arg\min_\xi \;\tfrac12\, r(\xi)^T W\, r(\xi),
$$

and they only differ in *what* the residual is:

| Front end | Residual $r(\xi)$ | Noise units |
|---|---|---|
| ICP (point-to-point / point-to-plane) | distance per matched point | metres ($\sigma_{\text{pt}}$) |
| Bundle adjustment / E-matrix VO | pixel reprojection error per landmark | pixels ($\sigma_{\text{pix}}$) |
| Direct VO / dense optical flow | photometric error per pixel | grey levels ($\sigma_{\text{int}}$) |

Under Gaussian noise on the residuals, the MLE covariance is the inverse Fisher information, which always collapses to $(J^T W J)^{-1}$ — regardless of whether $r$ measures distances, pixels, or intensities. That is the unifying principle: the *equation* depends only on the local noise model, not on the physical quantity being measured. The three sections below differ only in how they construct $r$, $J$, and $W$ for their particular sensor.

Wheel odometry (§1.5.2) is the odd one out — there is no front-end optimizer minimizing residuals, so $\Sigma$ comes from an empirically calibrated motion-noise model rather than an inverse Hessian.

---

#### 1.5.2 Wheel odometry

Wheel-odometry noise depends on how far the robot moved and how much it turned. The standard ("Thrun") motion-noise model parameterizes per-step covariance by traveled distance $\Delta d$ and turned angle $\Delta\theta$:

$$
\sigma_d^2 = \alpha_1 |\Delta d| + \alpha_2 |\Delta\theta|, \qquad
\sigma_\theta^2 = \alpha_3 |\Delta\theta| + \alpha_4 |\Delta d|
$$

The four $\alpha$ parameters are calibrated by driving a known path and fitting $\Sigma$ to the residuals. Wheel slip makes $\sigma_\theta$ much larger on slippery floors → values are platform-specific and surface-specific.

If an edge spans multiple odometry steps, propagate per-step $\Sigma_k$ through the chain composition:

$$
\Sigma_{ij} \;\approx\; \sum_{k=i}^{j-1} J_k\, \Sigma_k\, J_k^T
$$

where $J_k$ is the Jacobian of the chained motion model at step $k$.

#### 1.5.3 Laser scan matching (ICP / NDT)

A scan matcher returns a relative pose plus a covariance read directly off the matcher's own Hessian at the converged solution:

$$
\Sigma \;\approx\; \big(J^T W\, J\big)^{-1}
$$

where $J$ is the Jacobian of the per-correspondence residual w.r.t. the pose, and $W$ is the per-correspondence weight (often identity). For ICP this is **Censi's closed-form covariance** (Censi 2007); for NDT, it's the Hessian of the negative-log-likelihood at the matched pose.

Caveats:

* The inverse-Hessian estimate is **optimistic** — it assumes correct correspondences and Gaussian noise. Real scans break both.
* In long, featureless corridors, the Hessian is rank-deficient along the corridor axis → covariance becomes huge in that direction (correctly capturing that the matcher cannot localize you longitudinally). This is one place where a **full** (non-diagonal) $\Sigma$ matters.
* Many SLAM stacks scale Censi's covariance by an empirical factor (e.g., ×4) or fall back to fixed values when the Hessian is ill-conditioned.

#### 1.5.4 Visual odometry — Essential / Fundamental matrix, triangulation, BA

For a camera-based front end (Essential-matrix decomposition, PnP, two-view triangulation, or full bundle adjustment), the relative-pose covariance comes from propagating **pixel noise** through the estimator:

$$
\Sigma_{\text{pose}} \;\approx\; \big(J^T \Sigma_{\text{pix}}^{-1} J\big)^{-1}
$$

where $J$ is the Jacobian of the reprojection residuals w.r.t. the relative pose, and $\Sigma_{\text{pix}}$ is the pixel-noise covariance (typically $\sigma_{\text{pix}} \in [0.5, 2]$ px, often isotropic).

What dominates the result:

* **Number of inliers** — more correspondences shrink $\Sigma$ roughly as $1/N$.
* **Spatial distribution** — features clustered in one image region under-constrain rotation; spread-out features tighten it.
* **Baseline / depth ratio** — short baseline + far points yields a poor translation estimate, with large $\sigma_t$ along the optical axis.
* **Monocular scale ambiguity** — pure monocular VO has unobservable translation magnitude. Either resolve the scale (IMU, stereo, known object size) or assign $\sigma_{|t|} \to \infty$ in that direction so the optimizer ignores it.

ORB-SLAM, VINS-Mono, OpenVSLAM and similar systems compute this covariance from the local-BA Hessian at convergence and pass it forward as the edge information.

#### 1.5.5 Optical flow

If the front end is dense or sparse optical flow that you turn into a pose estimate (DSO-style direct alignment, or a 5-point solver on flow vectors), the pipeline is the same form as above:

$$
\Sigma_{\text{pose}} = \big(J^T \Sigma_{\text{flow}}^{-1} J\big)^{-1}
$$

with $\Sigma_{\text{flow}}$ encoding per-pixel flow noise. Direct methods often estimate $\sigma_{\text{flow}}$ from the photometric residual itself (variance of the brightness-constancy error), which automatically downweights low-texture regions: a blank wall produces high residual → high $\sigma_{\text{flow}}$ → low information weight, exactly the right behavior.

#### 1.5.6 Practical defaults when in doubt

If you cannot compute $\Sigma$ from the front end, defensible starting points:

| Source | $\sigma_{x,y}$ (m) | $\sigma_\theta$ (rad) |
|---|---|---|
| Wheel odometry, per meter traveled | 0.05 | 0.01 |
| 2D lidar scan match (good geometry) | 0.02 | 0.01 |
| Visual odometry (good baseline, many features) | 0.05 | 0.02 |
| Loop closure (lidar / appearance match) | 0.05 – 0.10 | 0.02 – 0.05 |

Tune from there. Doubling every $\Sigma$ in the graph yields the same MAP estimate, so what matters is the **ratio** between edge types — odometry should usually be looser than a verified loop closure, and dramatically tighter than a possibly-wrong loop closure.

---

### 1.6 Where $\Omega$ gets used computationally

When you linearize each residual:

$$
e_{ij}(X) \approx e_{ij}(X^{(k)}) + J_{ij}\delta
$$

the normal equations accumulate weights:

$$
H = \sum J_{ij}^T \Omega_{ij} J_{ij}, \quad b = \sum J_{ij}^T \Omega_{ij} e_{ij}
$$

So $\Omega$ directly controls how much each edge contributes to the Hessian $H$ and gradient $b$.

---

### 1.7 One important warning: loop-closure outliers

If a loop closure is wrong, a large $\Omega$ can break the solution. In real systems you often use:

* a **robust kernel** (Huber/Cauchy) multiplying the cost, and/or
* loop-closure verification (RANSAC, consistency checks)

---

## 2. Quadratic Approximation of the Cost
Starting point (single edge $(i,j)$):

$$
\min_{\delta x}\ \left\| \bar e_{ij} - J_{ij}\delta x \right\|_{\Omega_{ij}}^2 \;=\; \min_{\delta x}\ (\bar e_{ij} - J_{ij}\delta x)^T \Omega_{ij} (\bar e_{ij} - J_{ij}\delta x)
$$

It's convenient to include the $\frac{1}{2}$ (it doesn't change the minimizer):

$$
\min_{\delta x}\ E(\delta x),\quad E(\delta x)=\frac{1}{2}(\bar e_{ij} - J_{ij}\delta x)^T \Omega_{ij} (\bar e_{ij} - J_{ij}\delta x)
$$

---



Expand the product:

$$
E(\delta x) = \frac{1}{2}\left( \bar e_{ij}^T\Omega_{ij}\bar e_{ij} - 2\,\delta x^T J_{ij}^T\Omega_{ij}\bar e_{ij} + \delta x^T J_{ij}^T\Omega_{ij}J_{ij}\delta x \right)
$$

The first term $\frac{1}{2}\bar e_{ij}^T\Omega_{ij}\bar e_{ij}$ is constant w.r.t. $\delta x$.

---

### 2.1 Take derivative w.r.t. $\delta x$

Use the standard identities (with $A$ symmetric):

* $\frac{\partial}{\partial x}(x^T A x) = (A + A^T)x = 2Ax$
* $\frac{\partial}{\partial x}(x^T c) = c$

Here $J_{ij}^T\Omega_{ij}J_{ij}$ is symmetric if $\Omega_{ij}$ is symmetric (it is).

So:

$$
\frac{\partial E}{\partial \delta x} = - J_{ij}^T\Omega_{ij}\bar e_{ij} + J_{ij}^T\Omega_{ij}J_{ij}\,\delta x
$$

---

### 2.2 Set derivative to zero (normal equations)

$$
- J_{ij}^T\Omega_{ij}\bar e_{ij} + J_{ij}^T\Omega_{ij}J_{ij}\,\delta x = 0
$$

Rearrange:

$$
\left(J_{ij}^T\Omega_{ij}J_{ij}\right)\delta x = J_{ij}^T\Omega_{ij}\bar e_{ij}
$$

Define:

$$
H_{ij} = J_{ij}^T\Omega_{ij}J_{ij}, \quad b_{ij} = J_{ij}^T\Omega_{ij}\bar e_{ij}
$$

Then:

$$
H_{ij}\,\delta x = b_{ij}
$$

---

### 2.3 Optional: relate $J_{ij}$ to $J_i, J_j$

If

$$
\delta x = \begin{bmatrix} \Delta x_i \\ \Delta x_j \end{bmatrix}, \quad J_{ij} = \begin{bmatrix} J_i & J_j \end{bmatrix}
$$

then the blocks are:

$$
H_{ii}=J_i^T\Omega J_i,\quad H_{ij}=J_i^T\Omega J_j,\quad H_{ji}=J_j^T\Omega J_i,\quad H_{jj}=J_j^T\Omega J_j
$$

$$
b_i = J_i^T\Omega \bar e,\quad b_j = J_j^T\Omega \bar e
$$

That's the exact per-edge contribution you accumulate into the global sparse system $H\Delta x=b$.

## 3. Iterative Optimization Main Loop (Gauss–Newton / Levenberg–Marquardt)



### 3.1 Initialize linear system

Let there be $N$ pose nodes, each with a 3-DOF state:

$$
X_i = (x_i, y_i, \theta_i), \quad i = 0,\dots,N-1
$$


$$
H \leftarrow 0,\quad b \leftarrow 0
$$


$$
b \in \mathbb{R}^{3N}
$$

$$
\boxed{
b = \mathbf{0}_{3N \times 1}
}
$$



### 3.2 For each constraint (edge) $(i,j)$ — compute predicted relative pose


$$
\hat z_{ij} = h(X_i^k, X_j^k)
$$



### 3.3 Compute residual

$$
e_{ij} = z_{ij} - \hat z_{ij}
$$

(Angle component wrapped to $(-\pi,\pi]$)



### 3.4 Compute Jacobian blocks

$$
J_i = \frac{\partial h(X_i,X_j)}{\partial X_i}, \quad J_j = \frac{\partial h(X_i,X_j)}{\partial X_j}
$$

So that

$$
J_{ij} = \begin{bmatrix} J_i & J_j \end{bmatrix}
$$



### 3.5 Update residual vector $b$

$$
b_i \mathrel{+}= J_i^T \Omega_{ij} e_{ij}
$$

$$
b_j \mathrel{+}= J_j^T \Omega_{ij} e_{ij}
$$

(Only entries corresponding to poses $i$ and $j$ are non-zero)





$$
b \in \mathbb{R}^{3N}
$$

and it is stacked as:

$$
b =
\begin{bmatrix}
b_0 \\
b_1 \\
\vdots \\
b_{N-1}
\end{bmatrix}
$$

where **each block corresponds to one pose**.



Each pose $X_i = (x_i, y_i, \theta_i)$ has **3 DOF**, therefore:

$$
\boxed{
b_i \in \mathbb{R}^{3 \times 1}
}
$$

Explicitly:

$$
b_i =
\begin{bmatrix}
b_{i,x} \\
b_{i,y} \\
b_{i,\theta}
\end{bmatrix}
$$

So:

* $b_i$ is **the slice of the global vector $b$** associated with pose $i$
* It stores how much pose $i$ is being "pulled" by all constraints connected to it





### 3.6 Update information matrix $H$

$$
H_{ii} \mathrel{+}= J_i^T \Omega_{ij} J_i
$$

$$
H_{ij} \mathrel{+}= J_i^T \Omega_{ij} J_j
$$

$$
H_{ji} \mathrel{+}= J_j^T \Omega_{ij} J_i
$$

$$
H_{jj} \mathrel{+}= J_j^T \Omega_{ij} J_j
$$

(All other blocks remain zero → **sparse matrix**)



### 3.7 Apply gauge fixing

Fix one pose (e.g. $X_0$) or add a strong prior to make $H$ full rank.



### 3.8 Solve the linear system

$$
H \Delta x = b
$$

where

$$
\Delta x = \begin{bmatrix} \Delta x_0^T & \Delta x_1^T & \cdots & \Delta x_N^T \end{bmatrix}^T
$$



### 3.9 Update poses

For each node $i$:

$$
X_i^{k+1} = X_i^k \oplus \Delta x_i
$$

(where $\oplus$ denotes pose composition / retraction on $SE(2)$)

---

### 3.10 Check convergence

Stop if:

$$
\|\Delta x\| < \varepsilon \quad \text{or} \quad \Delta E < \varepsilon
$$

---

### 3.11 (Optional) Levenberg–Marquardt damping

Replace step 4 by:

$$
(H + \lambda I)\Delta x = b
$$

and adapt $\lambda$.


